# A4I 2026 kickoff demo—stage the full orbital catalogue (maintainers only)

**What this does.** Pulls the **full current public catalogue** of orbital elements from
**Space-Track.org** with Patrick's own account, pulls the satellite catalogue (SATCAT) from CelesTrak,
validates both, and stages them in the class demos bucket:

```
gs://class-demo/a4i-2026/demo-orbital-conjunction/catalog/<UTC timestamp>/   ← kept forever
gs://class-demo/a4i-2026/demo-orbital-conjunction/catalog/latest/            ← what everything reads
```

**Why Space-Track and not CelesTrak.** CelesTrak's account-free queries reach about 29,000 objects and
cannot return dead satellites in bulk—roughly a fifth of the band our fictional fleet flies in.
Space-Track's single full-catalogue query returns everything with current elements.

**Rights.** USSPACECOM *"has provided express blanket approval for transfer/redistribution of basic SSA
data and services accessed via www.Space-Track.org conditioned on appropriate citation."* Basic SSA data
is TLEs/OMMs, SATCAT and decay data—exactly and only what this stages. Every snapshot carries a
`CITATION.txt`. Nothing here touches conjunction messages or covariance, which that approval does not cover.

**Credentials are typed, used once, and discarded.** Nothing is written to disk, printed, or kept in the
notebook. Optionally, store them in Secret Manager and set `SECRET_ID` instead.

**Be kind to Space-Track.** They ask for the full-catalogue query **at most once an hour**, at a minute
that is not the top or bottom of the hour, and fewer than 30 requests a minute. This notebook makes three
requests (login, one query, logout) plus one to read the field list. Re-running inside the hour reuses the
cached pull.

**To refresh before an event:** open this notebook, Run all, type the credentials. About two minutes.

In [ ]:
import sys, subprocess, importlib.metadata as md
try: md.version("sgp4")
except md.PackageNotFoundError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sgp4"], check=True)

import os, io, json, time, hashlib, socket, platform, datetime as dt
from pathlib import Path
import requests, numpy as np, pandas as pd
import google.auth

_, PROJECT = google.auth.default(); PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT") or PROJECT
DEST_BASE = "gs://class-demo/a4i-2026/demo-orbital-conjunction/catalog"
EPOCH_WINDOW_DAYS = 30        # Space-Track's docs suggest 10 for "propagable"; 30 matches CelesTrak and keeps
                              # the aged objects visible so the demo can flag element age rather than hide it
SECRET_ID = ""                # optional: a Secret Manager secret holding {"identity": "...", "password": "..."}
WORK = Path("stage_work"); WORK.mkdir(exist_ok=True)
ST = "https://www.space-track.org"
GP_QUERY = (f"/basicspacedata/query/class/gp/decay_date/null-val/epoch/%3Enow-{EPOCH_WINDOW_DAYS}"
            f"/orderby/norad_cat_id/format/csv")
SATCAT_URL = "https://celestrak.org/pub/satcat.csv"
UA = {"User-Agent": "A4I2026-demo-stage/1.0 (ROI Training)"}
STAMP = f"{dt.datetime.now(dt.timezone.utc):%Y%m%dT%H%MZ}"

CHECKS = []
def check(name, ok, detail=""):
    CHECKS.append(("PASS" if ok else "FAIL", name, str(detail))); print("PASS" if ok else "FAIL", "·", name, "·", detail)
def note(name, ok, detail=""):
    CHECKS.append(("OK" if ok else "WARN", name, str(detail))); print("OK" if ok else "WARN", "·", name, "·", detail)
print({"project": PROJECT, "python": sys.version.split()[0], "host": socket.gethostname(), "stamp": STAMP})

## 1 — Pull the full catalogue from Space-Track

Login is a `POST` to `ajaxauth/login` with `identity` and `password`; a failed login answers with
`{"Login": "Failed"}` rather than an HTTP error, so we test the body. The field list comes from
Space-Track's own `modeldef` endpoint rather than from memory.

In [ ]:
cache_gp = WORK / "gp_full.csv"; cache_meta = WORK / "gp_full.json"
fresh = cache_gp.exists() and cache_meta.exists() and \
        (time.time() - json.loads(cache_meta.read_text())["fetched_epoch"]) < 3600
if fresh:
    GP_META = json.loads(cache_meta.read_text()); print("reusing a pull from", GP_META["fetched_utc"])
else:
    if SECRET_ID:
        from google.cloud import secretmanager
        sm = secretmanager.SecretManagerServiceClient()
        creds = json.loads(sm.access_secret_version(name=f"projects/{PROJECT}/secrets/{SECRET_ID}/versions/latest").payload.data)
    else:
        from getpass import getpass
        creds = {"identity": input("Space-Track username (email): ").strip(), "password": getpass("Space-Track password: ")}
    s = requests.Session(); s.headers.update(UA)
    try:
        r = s.post(ST + "/ajaxauth/login", data=creds, timeout=60)
        failed = r.status_code != 200 or '"Login":"Failed"' in r.text.replace(" ", "")
        if failed:
            raise RuntimeError(f"Space-Track login failed (HTTP {r.status_code}). Check the credentials; do not retry in a loop.")
        mdl = s.get(ST + "/basicspacedata/modeldef/class/gp", timeout=60)   # JSON: {"data": [{"Field": ..., "Type": ...}]}
        try: FIELDS = [f"{f['Field']}:{f['Type']}" for f in mdl.json()["data"]]
        except Exception: FIELDS = [f"modeldef unreadable: HTTP {mdl.status_code} {mdl.text[:120]!r}"]
        t = time.perf_counter()
        r = s.get(ST + GP_QUERY, timeout=300)
        secs = round(time.perf_counter() - t, 1)
        if r.status_code != 200 or "NORAD_CAT_ID" not in r.text[:2000]:
            raise RuntimeError(f"GP query failed: HTTP {r.status_code}: {r.text[:300]!r}")
        cache_gp.write_text(r.text)
        GP_META = {"url": ST + GP_QUERY, "status": r.status_code, "bytes": len(r.content), "seconds": secs,
                   "fetched_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
                   "fetched_epoch": time.time(), "modeldef_fields": FIELDS}
        cache_meta.write_text(json.dumps(GP_META))
    finally:
        try: s.get(ST + "/ajaxauth/logout", timeout=30)
        except Exception: pass
        creds = None
print({k: v for k, v in GP_META.items() if k != "modeldef_fields"})
print("GP fields per modeldef:", GP_META.get("modeldef_fields"))

## 2 — The satellite catalogue, from CelesTrak

SATCAT names every catalogued object, alive or dead, with its type and status. CelesTrak serves it
without an account. One download; cached for a day.

In [ ]:
cache_sc = WORK / "satcat.csv"; meta_sc = WORK / "satcat.json"
if cache_sc.exists() and meta_sc.exists() and (time.time() - json.loads(meta_sc.read_text())["fetched_epoch"]) < 86400:
    SC_META = json.loads(meta_sc.read_text()); print("reusing SATCAT from", SC_META["fetched_utc"])
else:
    r = requests.get(SATCAT_URL, headers=UA, timeout=120)
    if r.status_code != 200 or not r.text.startswith("OBJECT_NAME"):
        raise RuntimeError(f"SATCAT: HTTP {r.status_code}: {r.text[:300]!r} — CelesTrak asks clients to stop on any non-200.")
    cache_sc.write_text(r.text)
    SC_META = {"url": SATCAT_URL, "status": 200, "bytes": len(r.content), "last_modified": r.headers.get("last-modified"),
               "fetched_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"), "fetched_epoch": time.time()}
    meta_sc.write_text(json.dumps(SC_META))
print(SC_META)

## 3 — Validate before staging

`check()` is our pipeline (fatal to the publish step); `note()` is what the publisher handed us (warns,
prints the rows). The decisive check: **every row must initialise in SGP4**, because a row that cannot
be propagated is a row the screen silently skips.

In [ ]:
from sgp4.api import Satrec
from sgp4 import omm
GP = pd.read_csv(cache_gp, dtype=str, keep_default_na=False)
SAT = pd.read_csv(cache_sc, dtype=str, keep_default_na=False)
OMM_REQ = ["OBJECT_NAME","OBJECT_ID","EPOCH","MEAN_MOTION","ECCENTRICITY","INCLINATION","RA_OF_ASC_NODE",
           "ARG_OF_PERICENTER","MEAN_ANOMALY","EPHEMERIS_TYPE","CLASSIFICATION_TYPE","NORAD_CAT_ID",
           "ELEMENT_SET_NO","REV_AT_EPOCH","BSTAR","MEAN_MOTION_DOT","MEAN_MOTION_DDOT"]
missing_cols = [c for c in OMM_REQ if c not in GP.columns]
check("GP carries every OMM field SGP4 needs", not missing_cols, missing_cols or f"{len(GP.columns)} columns")
print("GP columns:", list(GP.columns))
dups = GP["NORAD_CAT_ID"].duplicated().sum()
check("one row per object", dups == 0, f"{dups} duplicate NORAD_CAT_IDs")

def omm_fields(rec):
    # sgp4.omm.initialize parses EPOCH with '%Y-%m-%dT%H:%M:%S.%f' and int()s two counters; normalise both.
    f = {k: rec[k] for k in OMM_REQ}
    if "." not in f["EPOCH"]: f["EPOCH"] += ".000000"
    for k in ("ELEMENT_SET_NO", "REV_AT_EPOCH", "EPHEMERIS_TYPE"):
        f[k] = f[k] if f[k].strip().lstrip("-").isdigit() else "0"
    f["CLASSIFICATION_TYPE"] = f["CLASSIFICATION_TYPE"] or "U"
    return f

bad = []
for rec in GP[OMM_REQ].to_dict("records"):
    try:
        s = Satrec(); omm.initialize(s, omm_fields(rec))
        if s.error != 0: bad.append((rec["NORAD_CAT_ID"], rec["OBJECT_NAME"], f"sgp4 error {s.error}"))
    except Exception as e:
        bad.append((rec["NORAD_CAT_ID"], rec["OBJECT_NAME"], f"{type(e).__name__}: {e}"))
check("every row initialises in SGP4", len(bad) <= 0.001 * len(GP), f"{len(bad)} of {len(GP):,}: {bad[:5]}")

ep = pd.to_datetime(GP["EPOCH"], utc=True, errors="coerce", format="ISO8601")
age = (pd.Timestamp.now(tz="UTC") - ep).dt.total_seconds() / 86400
check("every EPOCH parses", ep.notna().all(), int(ep.isna().sum()))
on = SAT[(SAT["ORBIT_CENTER"] == "EA") & (SAT["DECAY_DATE"] == "")]
onel = on[on["DATA_STATUS_CODE"] != "NEA"]
cov_missing = onel[~onel["NORAD_CAT_ID"].isin(set(GP["NORAD_CAT_ID"]))]
note("full catalogue covers SATCAT's on-orbit objects with elements", len(cov_missing) / len(onel) < 0.05,
     f"{len(cov_missing):,} of {len(onel):,} not in the pull; by type {cov_missing['OBJECT_TYPE'].value_counts().to_dict()}")
starlink = GP["OBJECT_NAME"].str.startswith("STARLINK").sum()
SUMMARY = {
    "gp_rows": len(GP), "gp_bytes": GP_META["bytes"], "six_digit_ids": int((pd.to_numeric(GP["NORAD_CAT_ID"]) >= 100000).sum()),
    "epoch_age_days": {q: round(float(age.quantile(q)), 2) for q in (0.1, 0.5, 0.9, 0.99, 1.0)},
    "object_type": GP["OBJECT_TYPE"].value_counts().to_dict() if "OBJECT_TYPE" in GP.columns else "no OBJECT_TYPE column",
    "starlink_rows": int(starlink), "satcat_rows": len(SAT), "satcat_on_orbit_with_elements": len(onel),
    "not_in_pull": len(cov_missing), "sgp4_init_failures": len(bad),
}
print(json.dumps(SUMMARY, indent=1))

## 4 — Stage it

Written once to a timestamped folder that is never overwritten, then copied to `latest/`. A manifest
records where every file came from, when, how big it is and its SHA-256, so any number the demo shows
can be traced to the exact pull it came from.

In [ ]:
fatal = [c for c in CHECKS if c[0] == "FAIL"]
assert not fatal, f"Not staging: {len(fatal)} FAIL checks above. Fix the pipeline, not the data."

CITATION = (
    "Orbital data: U.S. Space Command (USSPACECOM), via Space-Track.org; satellite catalogue via CelesTrak (celestrak.org).\n"
    "Redistributed under USSPACECOM's express blanket approval for basic SSA data (TLEs/OMMs, SATCAT, decay data),\n"
    "which is conditioned on appropriate citation. No warranty of accuracy or completeness.\n"
    f"Pulled {GP_META['fetched_utc']} (GP) and {SC_META['fetched_utc']} (SATCAT).\n")
(WORK / "CITATION.txt").write_text(CITATION)
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
MANIFEST = {
    "stamp": STAMP, "project": PROJECT,
    "gp_full.csv": {"source": GP_META["url"], "fetched_utc": GP_META["fetched_utc"], "rows": len(GP), "sha256": sha(cache_gp),
                    "epoch_window_days": EPOCH_WINDOW_DAYS, "fields": list(GP.columns)},
    "satcat.csv": {"source": SC_META["url"], "fetched_utc": SC_META["fetched_utc"], "last_modified": SC_META.get("last_modified"),
                   "rows": len(SAT), "sha256": sha(cache_sc)},
    "summary": SUMMARY, "checks": [" | ".join(c) for c in CHECKS], "citation": CITATION,
}
(WORK / "manifest.json").write_text(json.dumps(MANIFEST, indent=1))

files = [str(WORK / f) for f in ("gp_full.csv", "satcat.csv", "manifest.json", "CITATION.txt")]
dest = f"{DEST_BASE}/{STAMP}/"
r1 = subprocess.run(["gcloud", "storage", "cp", *files, dest], capture_output=True, text=True)
check("uploaded the timestamped snapshot", r1.returncode == 0, dest if r1.returncode == 0 else r1.stderr[-400:])
if r1.returncode == 0:
    r2 = subprocess.run(["gcloud", "storage", "cp", *files, f"{DEST_BASE}/latest/"], capture_output=True, text=True)
    check("copied to latest/", r2.returncode == 0, r2.stderr[-400:] if r2.returncode else f"{DEST_BASE}/latest/")
    back = subprocess.run(["gcloud", "storage", "cat", f"{DEST_BASE}/latest/manifest.json"], capture_output=True, text=True)
    ok = back.returncode == 0 and json.loads(back.stdout)["stamp"] == STAMP
    check("latest/manifest.json reads back as this run", ok, back.stderr[-300:] if back.returncode else STAMP)

## Diagnostic block—copy everything this cell prints and paste it back

In [ ]:
print("===== A4I DEMO CATALOGUE STAGE — DIAGNOSTIC BLOCK =====")
print(json.dumps({"stamp": STAMP, "dest": f"{DEST_BASE}/{STAMP}/", "gp_meta": {k: v for k, v in GP_META.items() if k != "modeldef_fields"},
                  "gp_fields": GP_META.get("modeldef_fields"), "satcat": SC_META, "summary": SUMMARY,
                  "checks": [" | ".join(c) for c in CHECKS]}, indent=1, default=str))
print("===== END =====")